<a href="https://colab.research.google.com/github/Hiteshyc/DSFFN-Deepfake-Detection/blob/main/dsffn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# STEP 1: SETUP AND DATASET DOWNLOAD
print("--- Step 1: Setting up environment and downloading TRAINING dataset ---")

!pip install kaggle -q

import os
import glob
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import accuracy_score, roc_auc_score
from tqdm import tqdm
import shutil
from google.colab import files

# --- Upload your kaggle.json ---
print("\nPlease upload your kaggle.json file:")
try:
    files.upload()
except:
    print("Upload helper not available. Please upload kaggle.json to the Colab sidebar.")

# --- Configure Kaggle API ---
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Training/Validation dataset
print("\nDownloading image dataset (manjilkarki/deepfake-and-real-images)...")
!kaggle datasets download -d manjilkarki/deepfake-and-real-images -q
print("Download complete.")

print("Unzipping dataset...")
# -o overwrites files without asking
!unzip -o -q deepfake-and-real-images.zip -d /content/dataset
print("Unzip complete. Data is in /content/dataset")

# STEP 2: DEFINE THE MODELS
print("\n--- Step 2: Defining models ---")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Use EfficientNet-B0 and a consistent image size for all experiments
IMG_SIZE = 224

# --- Model 1: Spatial Only (Baseline) ---
class SpatialOnlyModel(nn.Module):
    def __init__(self):
        super(SpatialOnlyModel, self).__init__()
        self.base_model = models.efficientnet_b0(pretrained=True)
        num_ftrs = self.base_model.classifier[1].in_features
        # Replace the classifier for binary output
        self.base_model.classifier = nn.Sequential(
            nn.Dropout(p=0.2, inplace=True),
            nn.Linear(num_ftrs, 1)
        )

    def forward(self, x):
        return self.base_model(x)

# --- Model 2: Frequency Only (Baseline) ---
class FreqOnlyModel(nn.Module):
    def __init__(self):
        super(FreqOnlyModel, self).__init__()
        self.base_model = models.efficientnet_b0(pretrained=True)
        num_ftrs = self.base_model.classifier[1].in_features
        self.base_model.classifier = nn.Sequential(
            nn.Dropout(p=0.2, inplace=True),
            nn.Linear(num_ftrs, 1)
        )

    def forward(self, x):
        return self.base_model(x)

# --- Model 3: The Fused DSFFN (Proposed) ---
class DSFFN(nn.Module):
    def __init__(self):
        super(DSFFN, self).__init__()
        # 1. Spatial Stream
        self.spatial_stream = models.efficientnet_b0(pretrained=True)
        num_ftrs = self.spatial_stream.classifier[1].in_features
        # Remove the classifier head to get the feature vector
        self.spatial_stream.classifier = nn.Identity()

        # 2. Frequency Stream
        self.freq_stream = models.efficientnet_b0(pretrained=True)
        # Remove the classifier head
        self.freq_stream.classifier = nn.Identity()

        # 3. Fusion and Classifier Head
        fused_ftrs = num_ftrs * 2 # From concatenating both streams
        self.classifier_head = nn.Sequential(
            nn.Linear(fused_ftrs, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 1)
        )

    def forward(self, x_spatial, x_freq):
        f_spatial = self.spatial_stream(x_spatial)
        f_freq = self.freq_stream(x_freq)

        # Fuse features
        f_fused = torch.cat((f_spatial, f_freq), dim=1)

        output = self.classifier_head(f_fused)
        return output

print("Models defined: SpatialOnly, FreqOnly, DSFFN (Fused)")

# STEP 3: CUSTOM DATASET & DATALOADERS
print("\n--- Step 3: Setting up DataLoaders ---")

# Standard ImageNet transforms, applied to both streams
data_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class DualStreamDataset(Dataset):
    """
    Custom dataset that returns both SPATIAL (RGB) and FREQUENCY (Phase) inputs
    for a given image path.
    """
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []

        # Find all 'real' and 'fake' images
        # ***** THIS IS THE FIX *****
        # The folder names are capitalized: "Real" and "Fake"
        for label, class_name in enumerate(['Real', 'Fake']):
            class_dir = os.path.join(root_dir, class_name)
            if not os.path.isdir(class_dir):
                print(f"Warning: Directory not found: {class_dir}")
                continue

            for img_path in glob.glob(os.path.join(class_dir, '*.jpg')):
                self.image_paths.append(img_path)
                self.labels.append(label) # 0 for real, 1 for fake

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        """
        Loads an image and generates its spatial and frequency representations.
        """
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        try:
            img_pil = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Warning: Error loading image {img_path}: {e}. Returning dummy data.")
            # Return a dummy item on error
            return torch.zeros(3, IMG_SIZE, IMG_SIZE), torch.zeros(3, IMG_SIZE, IMG_SIZE), torch.tensor(0.0)

        # --- 1. Create SPATIAL input (RGB image) ---
        spatial_input = self.transform(img_pil)

        # --- 2. Create FREQUENCY input (Phase Spectrum) ---
        img_gray = np.array(img_pil.convert('L'))

        # Calculate 2D DFT and shift zero-freq component to center
        dft = cv2.dft(np.float32(img_gray), flags=cv2.DFT_COMPLEX_OUTPUT)
        dft_shift = np.fft.fftshift(dft)

        # Get phase (angle)
        _, phase = cv2.cartToPolar(dft_shift[:, :, 0], dft_shift[:, :, 1])

        # Normalize phase to 0-255 to look like an "image"
        phase_img = cv2.normalize(phase, None, 0, 255, cv2.NORM_MINMAX, cv2.CV_8U)

        # Convert 1-channel phase image to 3-channel to match EfficientNet input
        phase_pil = Image.fromarray(phase_img).convert('RGB')

        freq_input = self.transform(phase_pil)

        return spatial_input, freq_input, torch.tensor(label, dtype=torch.float32)

# --- Create DataLoaders ---
train_dir = '/content/dataset/Dataset/Train'
test_dir = '/content/dataset/Dataset/Test'

train_dataset = DualStreamDataset(train_dir, transform=data_transforms)
# The test_dataset acts as our validation set during training
test_dataset = DualStreamDataset(test_dir, transform=data_transforms)

# Lower BATCH_SIZE if you get CUDA OOM errors
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"DataLoaders created. Training images: {len(train_dataset)}, Testing (Validation) images: {len(test_dataset)}")

# STEP 4: TRAINING & TESTING FUNCTIONS
print("\n--- Step 4: Defining Training and Testing functions ---")

# Use BCEWithLogitsLoss because our models output raw logits (not a sigmoid)
criterion = nn.BCEWithLogitsLoss()

# --- Training Function ---
def train_model(model, loader, optimizer, model_type):
    """Performs one epoch of training."""
    model.train()
    running_loss = 0.0

    for spatial, freq, labels in tqdm(loader, desc=f"Training {model_type}"):
        spatial, freq, labels = spatial.to(device), freq.to(device), labels.to(device).unsqueeze(1)

        optimizer.zero_grad()

        if model_type == 'spatial':
            outputs = model(spatial)
        elif model_type == 'freq':
            outputs = model(freq)
        elif model_type == 'fused':
            outputs = model(spatial, freq)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(loader)
    print(f"Training Loss: {epoch_loss:.4f}")

# --- Testing Function ---
def test_model(model, loader, model_type, test_name="Testing"):
    """Evaluates the model and returns accuracy and AUC."""
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad(): # No gradients needed for evaluation
        for spatial, freq, labels in tqdm(loader, desc=f"{test_name} {model_type}"):
            spatial, freq, labels = spatial.to(device), freq.to(device), labels.to(device)

            if model_type == 'spatial':
                outputs = model(spatial)
            elif model_type == 'freq':
                outputs = model(freq)
            elif model_type == 'fused':
                outputs = model(spatial, freq)

            preds = torch.sigmoid(outputs).cpu().numpy()
            all_preds.extend(preds.flatten())
            all_labels.extend(labels.cpu().numpy().flatten())

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)

    # Handle case where a batch might only have one class
    try:
        acc = accuracy_score(all_labels, all_preds.round())
        auc = roc_auc_score(all_labels, all_preds)
    except ValueError:
        acc = 0.0
        auc = 0.0

    # Only print full results for final tests, not every epoch
    if "Epoch" not in test_name:
      print(f"RESULTS ({test_name}): Accuracy: {acc*100:.2f}%, AUC: {auc*100:.2f}%\n")

    return acc, auc

print("Functions defined.")

# STEP 5: RUN EXPERIMENTS (TEST 1: INTRA-DATASET)
print("\n--- Step 5: Starting Experiments (Test 1: Intra-Dataset) ---")

NUM_EPOCHS = 6 # As defined in the paper
results_intra = {} # Stores FINAL Test 1 results
epoch_auc_history = {'spatial': [], 'freq': [], 'fused': []} # Stores Figure 2 data

# --- Experiment 1: Spatial Only ---
print("\n=== EXPERIMENT 1: Training SpatialOnly Model ===")
model_spatial = SpatialOnlyModel().to(device)
optimizer_spatial = optim.Adam(model_spatial.parameters(), lr=1e-4)
for epoch in range(NUM_EPOCHS):
    print(f"--- Epoch {epoch+1}/{NUM_EPOCHS} ---")
    train_model(model_spatial, train_loader, optimizer_spatial, 'spatial')
    # Validate on the test set after each epoch
    acc, auc = test_model(model_spatial, test_loader, 'spatial', test_name=f"Epoch {epoch+1} Val")
    print(f"Epoch {epoch+1} Validation AUC: {auc*100:.2f}%")
    epoch_auc_history['spatial'].append(auc)
results_intra['spatial'] = (acc, auc) # Save the final epoch's results

# --- Experiment 2: Frequency Only ---
print("\n=== EXPERIMENT 2: Training FreqOnly Model ===")
model_freq = FreqOnlyModel().to(device)
optimizer_freq = optim.Adam(model_freq.parameters(), lr=1e-4)
for epoch in range(NUM_EPOCHS):
    print(f"--- Epoch {epoch+1}/{NUM_EPOCHS} ---")
    train_model(model_freq, train_loader, optimizer_freq, 'freq')
    acc, auc = test_model(model_freq, test_loader, 'freq', test_name=f"Epoch {epoch+1} Val")
    print(f"Epoch {epoch+1} Validation AUC: {auc*100:.2f}%")
    epoch_auc_history['freq'].append(auc)
results_intra['freq'] = (acc, auc)

# --- Experiment 3: Fused DSFFN Model ---
print("\n=== EXPERIMENT 3: Training Fused (DSFFN) Model ===")
model_fused = DSFFN().to(device)
optimizer_fused = optim.Adam(model_fused.parameters(), lr=1e-4)
for epoch in range(NUM_EPOCHS):
    print(f"--- Epoch {epoch+1}/{NUM_EPOCHS} ---")
    train_model(model_fused, train_loader, optimizer_fused, 'fused')
    acc, auc = test_model(model_fused, test_loader, 'fused', test_name=f"Epoch {epoch+1} Val")
    print(f"Epoch {epoch+1} Validation AUC: {auc*100:.2f}%")
    epoch_auc_history['fused'].append(auc)
results_intra['fused'] = (acc, auc)

print("\n--- Intra-Dataset Training & Validation Complete ---")
print(f"Final Intra-Dataset (Spatial)   Acc: {results_intra['spatial'][0]*100:.2f}%, AUC: {results_intra['spatial'][1]*100:.2f}%")
print(f"Final Intra-Dataset (Frequency) Acc: {results_intra['freq'][0]*100:.2f}%, AUC: {results_intra['freq'][1]*100:.2f}%")
print(f"Final Intra-Dataset (DSFFN)     Acc: {results_intra['fused'][0]*100:.2f}%, AUC: {results_intra['fused'][1]*100:.2f}%")


################################################################################
# STEP 6: RUN EXPERIMENTS (TEST 2: CROSS-DATASET GENERALIZATION)
################################################################################
print("\n--- Step 6: Running Cross-Dataset Generalization Test (Test 2) ---")
print("This is the core test of the paper's hypothesis.")

# --- 1. Download and Unzip the UNSEEN Dataset ---
print("\nDownloading UNSEEN dataset (ciplab/real-and-fake-face-detection)...")
!kaggle datasets download -d ciplab/real-and-fake-face-detection -q
print("Download complete.")

print("Unzipping unseen dataset...")
!unzip -o -q real-and-fake-face-detection.zip -d /content/ciplab_dataset
print("Unzip complete.")

# --- 2. Prepare the UNSEEN Dataset Loader ---
# This dataset has a different folder structure, so we rearrange it
ciplab_test_dir = "/content/ciplab_test_set"
ciplab_source_dir = "/content/ciplab_dataset/real_and_fake_face/real_and_fake_face"

os.makedirs(os.path.join(ciplab_test_dir, "Real"), exist_ok=True)
os.makedirs(os.path.join(ciplab_test_dir, "Fake"), exist_ok=True)

print("Moving test files to a clean directory structure...")
for f in glob.glob(os.path.join(ciplab_source_dir, "test_real", "*.jpg")):
    shutil.copy(f, os.path.join(ciplab_test_dir, "Real"))

for f in glob.glob(os.path.join(ciplab_source_dir, "test_fake", "*.jpg")):
    shutil.copy(f, os.path.join(ciplab_test_dir, "Fake"))

print("File move complete.")

ciplab_dataset = DualStreamDataset(ciplab_test_dir, transform=data_transforms)
ciplab_loader = DataLoader(ciplab_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Unseen test loader created. Images: {len(ciplab_dataset)}")

# --- 3. Evaluate the TRAINED Models on UNSEEN Data ---
# We use the models (model_spatial, model_freq, model_fused) that are
# *already trained* from Step 5. We DO NOT train them again.
results_cross = {}

print("\n=== EVALUATING SpatialOnly on UNSEEN Data ===")
acc, auc = test_model(model_spatial, ciplab_loader, 'spatial', test_name="Cross-Dataset")
results_cross['spatial'] = (acc, auc)

print("\n=== EVALUATING FreqOnly on UNSEEN Data ===")
acc, auc = test_model(model_freq, ciplab_loader, 'freq', test_name="Cross-Dataset")
results_cross['freq'] = (acc, auc)

print("\n=== EVALUATING Fused (DSFFN) on UNSEEN Data ===")
acc, auc = test_model(model_fused, ciplab_loader, 'fused', test_name="Cross-Dataset")
results_cross['fused'] = (acc, auc)

print("Cross-dataset evaluation complete.")


################################################################################
# STEP 7: PRINT AND SAVE FINAL COMPREHENSIVE RESULTS
################################################################################
print("\n\n--- === FINAL COMPREHENSIVE RESULTS === ---")

# --- 1. Prepare the output string ---
output_content = f"""
*** DSFFN Experiment Results Summary ***

This file contains the complete quantitative results from the paper
"DSFFN: Dual-Stream Forgery Fusion Network".


--- Test 1: Intra-Dataset Performance (Table 2) ---
Description: Final performance on the 'manjilkarki' (home) test set after {NUM_EPOCHS} epochs.
--------------------------------------------------
  Model             |  Accuracy  |  AUC
--------------------------------------------------
  Spatial Only      |  {results_intra['spatial'][0]*100:6.2f}%   | {results_intra['spatial'][1]*100:6.2f}%
  Frequency Only    |  {results_intra['freq'][0]*100:6.2f}%   | {results_intra['freq'][1]*100:6.2f}%
  DSFFN (Fused)     |  {results_intra['fused'][0]*100:6.2f}%   | {results_intra['fused'][1]*100:6.2f}%
--------------------------------------------------


--- Per-Epoch Validation AUC (Figure 2 Data) ---
Description: Validation AUC on the 'manjilkarki' test set after each epoch.
----------------------------------------------------------------------
  Epoch |  Spatial-Only   |  Frequency-Only |  DSFFN (Fused)
----------------------------------------------------------------------
"""

# Dynamically add the rows for the epoch table
for i in range(NUM_EPOCHS):
    epoch_num = i + 1
    spatial_auc = epoch_auc_history['spatial'][i] * 100
    freq_auc = epoch_auc_history['freq'][i] * 100
    fused_auc = epoch_auc_history['fused'][i] * 100
    output_content += f"    {epoch_num:<3} |  {spatial_auc:^13.2f}% |  {freq_auc:^15.2f}% |  {fused_auc:^14.2f}%\n"

output_content += "----------------------------------------------------------------------\n\n"

output_content += f"""--- Test 2: Cross-Dataset Generalization (Table 3) ---
Description: Models tested on UNSEEN 'ciplab' dataset.
-----------------------------------------------------------------------------------
  Model             |  Intra-Dataset  |  Cross-Dataset  |  Performance Drop
                    |  Accuracy (%)   |  Accuracy (%)   |  (Generalization Gap)
-----------------------------------------------------------------------------------
"""

# Dynamically add the rows for Table 3
for model_name in ['spatial', 'freq', 'fused']:
    intra_acc = results_intra[model_name][0] * 100
    cross_acc = results_cross[model_name][0] * 100
    perf_drop = cross_acc - intra_acc
    name_str = f"{model_name.capitalize()} Only".ljust(17) if 'spatial' in model_name or 'freq' in model_name else "DSFFN (Fused)".ljust(17)

    output_content += f"  {name_str} |  {intra_acc:^15.2f} |  {cross_acc:^15.2f} |  {perf_drop:^20.2f}%\n"

output_content += "-----------------------------------------------------------------------------------"


# --- 2. Print the output to the console ---
print(output_content)


# --- 3. Save the output to "output.txt" ---
try:
    with open("output.txt", "w") as f:
        f.write(output_content)
    print("\n\n--- === SUCCESS! === ---")
    print("Results have been saved to 'output.txt'.")
    print("You can find it in the Colab file browser on the left sidebar.")
except Exception as e:
    print(f"\n\n--- === ERROR === ---")
    print(f"An error occurred while saving the file: {e}")

--- Step 1: Setting up environment and downloading ALL datasets ---

Please upload your kaggle.json file:


Saving kaggle.json to kaggle (1).json
kaggle.json uploaded.

Dataset URL: https://www.kaggle.com/datasets/manjilkarki/deepfake-and-real-images
License(s): unknown
Download complete.
Unzipping dataset...
Unzip complete. Data is in /content/dataset

Dataset URL: https://www.kaggle.com/datasets/ciplab/real-and-fake-face-detection
License(s): CC-BY-NC-SA-4.0
Download complete.
Unzipping unseen dataset...
Unzip complete.
Moving test files to a clean directory structure...
File move complete.

--- Step 2: Defining models ---
Using device: cpu
Models defined: SpatialOnly, FreqOnly, DSFFN (Fused)

--- Step 3: Setting up DataLoaders ---
DataLoaders created. Training images: 140000, Testing (Validation) images: 20000

--- Step 4: Defining Training and Testing functions ---
Functions defined.

--- Step 5: Starting Experiments (Test 1: Intra-Dataset) ---

=== EXPERIMENT 1: Training SpatialOnly Model ===
--- Epoch 1/6 ---
Training spatial: 100%|██████████| 2188/2188 [08:31<00:00, 4.28it/s]
Training

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>